# Woche 2 - ML2 - Übung 2: Einfaches RAG mit PDFs

Entwickle einen Assistenten zur Turniervorbereitung für ein fiktives Esports-Coaching-Team.
Er soll Strategiefragen anhand der fünf Teamberichte in `data` beantworten.
Wenn sich eine Frage mit den Berichten nicht beantworten lässt, soll er dies mitteilen.

Wir lesen die PDFs ein, teilen den Text auf, suchen relevante Textpassagen und bitten ein Modell,
anhand dieser Passagen zu antworten. Das ist retrieval-augmented generation (RAG).

Arbeite einen Schritt nach dem anderen durch. Bearbeite die fünf `TODO`-Aufgaben und ergänze Notizen
an den angegebenen Stellen. Der übrige Code ist vorgegeben. Schliesse jedes TODO ab, bevor du die
Zellen ausführst, die davon abhängen.

## Was du üben wirst

- Entscheiden, wie Dokumente in Chunks aufgeteilt werden.
- Mit Embeddings und FAISS gefundene Textpassagen untersuchen.
- Einen Prompt schreiben, der die Antworten auf diese Passagen stützt.
- Fehlenden Kontext testen und vage Follow-up-Fragen verbessern.

Die Packages sind in `requirements.txt` aufgeführt. Wir verwenden `pypdf` zum Einlesen von PDFs
und `langchain-text-splitters` ausschliesslich zum Aufteilen von Text.

In [22]:
import os
from collections import Counter
from pathlib import Path

import faiss
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(".env")

True

In [23]:
DATA_DIR = Path("data")
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

if not PDF_FILES:
    raise FileNotFoundError("No PDFs found in data. Add PDF files first.")

PDF_FILES

[PosixPath('data/esports_overview_01_phantom_signal_valorant.pdf'),
 PosixPath('data/esports_overview_02_iron_tide_cs2.pdf'),
 PosixPath('data/esports_overview_03_neon_assembly_valorant.pdf'),
 PosixPath('data/esports_overview_04_vector_praetorians_cs2.pdf'),
 PosixPath('data/esports_overview_05_midnight_orbit_valorant.pdf')]

## Schritt 1 - PDFs einlesen

In [24]:
def read_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages).strip()


documents = []
for path in PDF_FILES:
    text = read_pdf_text(path)
    documents.append(
        {
            "source": path.name,
            "text": text,
            "char_count": len(text),
        }
    )

documents[:2]

incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(3)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling\nThis overview describes a fictional esports team created for teaching purposes.\nPhantom Signal started the split looking like a team assembled from highlight clips rather than\nshared habits. Their early matches were loud and frantic, with heroic moments that disguised\nweak structure. Round reviews showed the same pattern every series: good opening reads, poor\nmid-round adaptation, and visible emotional spikes after lost advantage rounds. The coaches did\nnot panic about tactics first. They treated communication discipline as the first lever, arguing\nthat no playbook survives if players stop hearing each other when pressure rises.\nThe in-game leader, Riven, had inherited an identity that was too scripted for the roster she\nactually had. In previous teams she won with set executions and tightly sequenced utility, but\nthis 

## Schritt 2 - Chunks festlegen (TODO 1)

Ein Chunk ist eine kurze Textpassage aus einem Dokument. Der Splitter misst seine Grösse in Zeichen.
Ein Overlap wiederholt einige Zeichen im nächsten Chunk, damit ein Satz möglichst keinen Kontext verliert.

Lege unten `CHUNK_SIZE` und `CHUNK_OVERLAP` fest. Der Overlap muss kleiner als die Chunk Size sein.
Ergänze einen kurzen Kommentar, der deine Wahl begründet.

Führe anschliessend den vorgegebenen Splitter aus und prüfe die Anzahl der Chunks. Sieh dir einige Chunks an:
Enthalten sie genügend Kontext, um verständlich zu sein?

In [25]:
# TODO 1: replace None with your chosen numbers of characters.
CHUNK_SIZE = 100
CHUNK_OVERLAP = 10

# Why these settings?
# ...

In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunk_records = []
for doc in documents:
    chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunk_records.append(
            {
                "source": doc["source"],
                "chunk_id": f"{doc['source']}_chunk_{i}",
                "chunk_text": chunk,
            }
        )

chunk_records[:10]

[{'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_0',
  'chunk_text': 'Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling'},
 {'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_1',
  'chunk_text': 'This overview describes a fictional esports team created for teaching purposes.'},
 {'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_2',
  'chunk_text': 'Phantom Signal started the split looking like a team assembled from highlight clips rather than'},
 {'source': 'esports_overview_01_phantom_signal_valorant.pdf',
  'chunk_id': 'esports_overview_01_phantom_signal_valorant.pdf_chunk_3',
  'chunk_text': 'shared habits. Their early matches were loud and frantic, with heroic moments that disguised'},
 {'source': 'esports_o

In [27]:
chunk_count_by_source = Counter(item["source"] for item in chunk_records)

print("Number of documents:", len(documents))
print("Number of chunks:", len(chunk_records))
print("\nChunks per source:")
for source, count in chunk_count_by_source.items():
    print(f"- {source}: {count}")

Number of documents: 5
Number of chunks: 263

Chunks per source:
- esports_overview_01_phantom_signal_valorant.pdf: 52
- esports_overview_02_iron_tide_cs2.pdf: 54
- esports_overview_03_neon_assembly_valorant.pdf: 54
- esports_overview_04_vector_praetorians_cs2.pdf: 52
- esports_overview_05_midnight_orbit_valorant.pdf: 51


## Schritt 3 - FAISS Retriever erstellen


### 3.1 Einmalig Embeddings für alle Chunks erstellen

Ein Embedding stellt eine Textpassage als Liste von Zahlen dar. Wir erstellen eines für jeden Chunk.
Die Query verwendet dasselbe Embedding Model, damit ihr Vektor mit den Vektoren der Chunks verglichen werden kann.

Führe diese Zelle einmal aus. Wenn du das Embedding Model oder die Chunks änderst, führe sie erneut aus
und erstelle den Index im nächsten Abschnitt neu.

In [28]:
embedding_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")


def embed_texts(texts, batch_size=64):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=batch,
        )
        vectors.extend(item.embedding for item in response.data)
    return np.array(vectors, dtype="float32")


chunk_texts = [item["chunk_text"] for item in chunk_records]
chunk_embeddings = embed_texts(chunk_texts)
faiss.normalize_L2(chunk_embeddings)
print("Chunk embeddings shape:", chunk_embeddings.shape)

Chunk embeddings shape: (263, 1536)


### 3.2 Den Vector Index erstellen

FAISS speichert Vektoren und ermöglicht eine effiziente Suche nach Nearest Neighbors.
Hier verwenden wir Inner Product auf normalisierten Vektoren, was sich wie Cosine Similarity verhält.


In [29]:
faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)
print("Vectors stored in FAISS:", faiss_index.ntotal)

Vectors stored in FAISS: 263


### 3.3 Textpassagen abrufen (TODO 2)

Die folgende Funktion erstellt ein Embedding der Frage, durchsucht FAISS und gibt passende Chunks zurück.
`top_k` legt die Anzahl der zurückzugebenden Treffer fest.

Führe zuerst die Zelle mit der Funktion aus. Führe anschliessend die beiden Beispielfragen mit `TOP_K = 3` aus.
Lies die drei Textpassagen, die für jede Frage zurückgegeben werden. Ein höherer Similarity Score bedeutet
eine grössere Übereinstimmung. Du musst aber weiterhin beurteilen, ob der Text die Frage beantwortet.

Ergänze eine kurze Beobachtung im markierten Kommentar am Ende der Beispielzelle.

In [30]:
def retrieve_chunks(query: str, top_k: int = 3):
    query_response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )
    query_vector = np.array([query_response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vector)

    k = min(top_k, len(chunk_texts))
    scores, indices = faiss_index.search(query_vector, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        item = chunk_records[int(idx)]
        results.append(
            {
                "source": item["source"],
                "chunk_id": item["chunk_id"],
                "similarity": float(score),
                "chunk_text": item["chunk_text"],
            }
        )

    return results

In [31]:
# TODO 2: run the two retrieval examples, then write your observation below.
# Start with three retrieved chunks per question.
TOP_K = 3

q1 = "How did Phantom Signal improve after early losses?"
q2 = "What role did communication protocols play in Neon Assembly's turnaround?"

for query in [q1, q2]:
    print("=" * 80)
    print("Query:", query)
    results = retrieve_chunks(query, top_k=TOP_K)
    for r in results:
        print(f"- {r['chunk_id']} | similarity={r['similarity']:.4f}")
        print(f"  {r['chunk_text'][:180]}...")

# Observation: do the passages discuss the right team and answer the question?
# ...

Query: How did Phantom Signal improve after early losses?
- esports_overview_01_phantom_signal_valorant.pdf_chunk_15 | similarity=0.6883
  A major turning point came after a painful loss where Phantom Signal led by six rounds and then...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_40 | similarity=0.6119
  By playoffs, Phantom Signal looked fundamentally different without appearing revolutionary. They...
- esports_overview_01_phantom_signal_valorant.pdf_chunk_0 | similarity=0.5947
  Phantom Signal (VALORANT): A comeback built on emotional control and clearer calling...
Query: What role did communication protocols play in Neon Assembly's turnaround?
- esports_overview_03_neon_assembly_valorant.pdf_chunk_22 | similarity=0.5370
  delayed, and interim responsibilities shifted weekly. Neon Assembly survived this by preserving...
- esports_overview_03_neon_assembly_valorant.pdf_chunk_2 | similarity=0.5326
  Neon Assembly rebuilt almost everything after a disappointing previous yea

## Schritt 4 - Den Prompt schreiben (TODO 3)

Ersetze die Anweisungszeile in `PROMPT_TEMPLATE` unten. Weise das Modell an:

- Als Assistent zur Turniervorbereitung zu handeln und Coaches knappe, praktische Ratschläge zu geben.
- Sich auf Strategie zu konzentrieren, etwa Kommunikation, Führung und Vorbereitung.
- Ausschliesslich den bereitgestellten Kontext zu verwenden.
- Genau `I don't know based on the provided context.` zu antworten, wenn dieser Kontext nicht ausreicht.

Behalte `{context}` und `{question}` im Template bei. Die vorgegebene Funktion zur Antwortgenerierung
füllt sie mit den abgerufenen Textpassagen und der Frage des Nutzers.

In [32]:
# TODO 3: write your instructions above Context.
PROMPT_TEMPLATE = """
You must act as an assistant in tournament preparation and provide coaches with concise, practical advice.
You should focus on strategy, such as communication, leadership and preparation.
You must only use the context provided.
If this context is insufficient, reply exactly with `I don't know based on the provided context.`.

Context:
{context}

Question:
{question}
"""

## Schritt 5 - Textpassagen abrufen und antworten

Die nächste Funktion fügt die abgerufenen Chunks in den Prompt ein und sendet ihn an OpenAI.
Sie gibt sowohl die Antwort als auch die verwendeten Textpassagen zurück.

Führe die Funktionsdefinition und anschliessend das Beispiel zu Iron Tide aus. Vergleiche die Antwort
mit den abgerufenen Textpassagen. Findest du Belege für ihre Aussagen?

In [33]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-6-luna")


def generate_answer(question: str, top_k: int = 3):
    retrieved = retrieve_chunks(question, top_k=top_k)
    context = "\n\n".join(item["chunk_text"] for item in retrieved)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
    )
    answer = response.output_text
    return answer, retrieved

In [34]:
test_question = "What strategic changes helped Iron Tide recover from bad starts?"
answer, used_chunks = generate_answer(test_question, top_k=TOP_K)

print("Question:", test_question)
print("\nAnswer:\n", answer)
print("\nRetrieved Chunks:")
for item in used_chunks:
    print(f"- {item['chunk_id']} | similarity={item['similarity']:.4f}")

Question: What strategic changes helped Iron Tide recover from bad starts?

Answer:
 I don't know based on the provided context.

Retrieved Chunks:
- esports_overview_02_iron_tide_cs2.pdf_chunk_26 | similarity=0.6263
- esports_overview_02_iron_tide_cs2.pdf_chunk_0 | similarity=0.6246
- esports_overview_02_iron_tide_cs2.pdf_chunk_52 | similarity=0.6006


## Schritt 6 - Failure Cases testen (TODO 4)

Führe die beiden folgenden Fragen aus: eine vage Frage und eine ohne Bezug zu den Berichten.
Lies sowohl die abgerufenen Textpassagen als auch die Antworten.

Ergänze eine kurze Notiz dazu, was in jedem Fall passiert ist. Hat das Modell auf
fehlende Informationen hingewiesen? Haben die Textpassagen tatsächlich geholfen, die Frage zu beantworten?

In [ ]:
vague_query = "Tell me more."
out_of_scope_query = "How can I optimize my home Wi-Fi router placement?"

for question in [vague_query, out_of_scope_query]:
    ans, ret = generate_answer(question, top_k=TOP_K)
    print("=" * 80)
    print("Question:", question)
    print("Answer:", ans)
    print("Top retrieved chunk IDs:", [item["chunk_id"] for item in ret])


# TODO 4: explain what happened and why.
# Vague question: ...
# Unrelated question: ...

Question: Tell me more.
Answer: I don't know based on the provided context.
Top retrieved chunk IDs: ['esports_overview_01_phantom_signal_valorant.pdf_chunk_27', 'esports_overview_05_midnight_orbit_valorant.pdf_chunk_7', 'esports_overview_01_phantom_signal_valorant.pdf_chunk_44']
Question: How can I optimize my home Wi-Fi router placement?
Answer: I don't know based on the provided context.
Top retrieved chunk IDs: ['esports_overview_01_phantom_signal_valorant.pdf_chunk_11', 'esports_overview_04_vector_praetorians_cs2.pdf_chunk_43', 'esports_overview_01_phantom_signal_valorant.pdf_chunk_30']


## Schritt 7 - Vage Follow-up-Fragen verbessern (TODO 5)

Eine Frage wie `Tell me more.` nennt kein Thema. Wir können die vorherige
Frage hinzufügen, damit der Retriever gezielter suchen kann.

Vervollständige unten `rewrite_query_if_vague`:

1. Wandle die Frage in Kleinbuchstaben um und entferne Leerzeichen am Anfang und Ende sowie abschliessende Satzzeichen.
   Die Methoden `.lower()`, `.strip()` und `.rstrip(".?!")` können dabei helfen.
2. Wenn sie mit einer Formulierung in `vague_patterns` übereinstimmt, gib
   `f"Provide more details about: {last_user_query}"` zurück.
3. Gib andernfalls die ursprüngliche Frage unverändert zurück.

Die Hilfsfunktion darunter ist vorgegeben. Sie verwendet deine umformulierte Frage mit der vorhandenen
Funktion `generate_answer`. Die vorherige Frage wird explizit übergeben; sie stammt
aus den unten stehenden Testfällen.

In [ ]:
# TODO 5: implement this function.
def rewrite_query_if_vague(question: str, last_user_query: str):
    vague_patterns = ["tell me more", "more", "what about that", "and this"]
    
    normalized = question.lower().strip().rstrip(".?!")

    if any(pattern in normalized for pattern in vague_patterns):
      return f"Provide more details about: {last_user_query}"

    return question

# Provided: use the rewritten question to retrieve passages and generate an answer.
def generate_answer_with_rewrite(question: str, last_user_query: str, top_k: int = 3):
    rewritten = rewrite_query_if_vague(question, last_user_query)
    answer, retrieved = generate_answer(rewritten, top_k=top_k)
    return rewritten, answer, retrieved


test_cases = [
    {
        "question": "Tell me more.",
        "last_user_query": "How did Midnight Orbit stabilize communication after coaching changes?",
    },
    {
        "question": "And this?",
        "last_user_query": "Why did Vector Praetorians simplify their strategy book?",
    },
]

records = []
for case in test_cases:
    question = case["question"]
    last_query = case["last_user_query"]

    base_answer, _ = generate_answer(question, top_k=TOP_K)
    rewritten_q, improved_answer, _ = generate_answer_with_rewrite(question, last_user_query=last_query, top_k=TOP_K)

    records.append(
        {
            "original_query": question,
            "rewritten_query": rewritten_q,
            "baseline_answer_preview": base_answer[:180],
            "improved_answer_preview": improved_answer[:180],
        }
    )

for row in records:
    print("=" * 80)
    print("Original query:", row["original_query"])
    print("Rewritten query:", row["rewritten_query"])
    print("Baseline preview:", row["baseline_answer_preview"])
    print("Improved preview:", row["improved_answer_preview"])


# TODO 5: compare the previews above.
# Did rewriting help? What limitation remains?
# ...

Original query: Tell me more.
Rewritten query: Provide more details about: How did Midnight Orbit stabilize communication after coaching changes?
Baseline preview: I don't know based on the provided context.
Improved preview: Assistant coaches stabilized communication by pausing broader strategic expansion and rebuilding the fundamentals around communication protocols. That kept team communication funct
Original query: And this?
Rewritten query: Provide more details about: Why did Vector Praetorians simplify their strategy book?
Baseline preview: I don't know based on the provided context.
Improved preview: I don't know based on the provided context.


## Bevor du abschliesst

- [ ] Begründe deine Wahl von Chunk Size und Overlap.
- [ ] Kommentiere das Retrieval für beide Beispielfragen.
- [ ] Schreibe und teste deinen auf den Kontext gestützten Prompt.
- [ ] Erläutere die beiden Failure Cases.
- [ ] Vergleiche die Antworten vor und nach dem Query Rewriting.
- [ ] Bearbeite die Reflexion unten.

Optionale Erweiterung: Verfeinere deinen Prompt und wiederhole einen Failure Case.
Die Musterlösung in `ml2_exercise2_solution.ipynb` demonstriert Query Rewriting.

## Abschliessende Reflexion

Antworte kurz und verwende insgesamt höchstens acht Zeilen.

1. Was hat in deiner Pipeline am besten funktioniert?
2. Was war der grösste Failure Mode?
3. Welche Designentscheidung hatte den grössten Einfluss?
4. Welche Einschränkung ergibt sich bei der Verwendung unvollständiger oder verzerrter Berichte?

Deine Reflexion: ...